In [ ]:
import torch
from PIL import Image, ImageOps
from transformers import DetrImageProcessor, DetrForObjectDetection
import numpy as np 

def image_detection(frame) :
    image = Image.open(frame)
    image = ImageOps.exif_transpose(image)
    image = image.convert("RGB")

    processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50", revision="no_timm")
    model = DetrForObjectDetection.from_pretrained("isalia99/detr-resnet-50-sku110k")
    model.eval()

    inputs = processor(images=image, return_tensors="pt")
     with torch.no_grad():
        outputs = model(**inputs)


    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.7)[0]
    for box in results["boxes"]:
         box_np = box.cpu().numpy().astype(int)  
         bbox.append(box_np)

return bbox 


14 174 49 196
306 34 315 61
216 260 245 271
161 29 170 61
125 31 137 60
59 256 92 274
440 287 471 303
283 345 301 374
115 34 124 62
27 31 40 61
421 32 431 60
438 287 470 303
541 97 565 110
533 292 552 330
400 266 421 290
529 83 538 111
247 34 258 62
207 263 229 274
147 84 159 112
444 132 476 151
460 32 469 61
278 80 301 104
202 208 211 234
19 251 34 277
149 164 179 183
206 34 215 62
305 171 329 198
208 183 239 194
144 85 156 112
450 29 460 61
148 260 177 269
18 298 45 310
155 83 169 112
533 135 564 145
513 83 528 110
213 334 239 346
437 248 451 276
359 341 387 356
100 220 119 233
19 341 47 374
549 34 561 61
213 206 224 233
537 223 566 234
247 33 258 62
442 344 469 374
504 207 520 234
359 337 387 351
306 201 333 232
310 342 331 375
102 81 127 111
372 33 384 61
8 250 19 277
108 308 131 336
326 32 336 61
102 220 121 233
175 32 185 62
113 36 122 62
359 316 385 335
451 313 467 331
15 31 28 60
398 32 409 60
278 167 302 197
541 97 566 110
366 245 393 276
168 217 194 234
232 33 244 62
206 262 

In [ ]:
import cv2 
import pandas as pd 
import time 
import csv 
import mediapipe as mp 
from ultralytics import YOLO 


def reset_csv():
    with open('data.csv' , 'w') as new_file   
        feild  = ['bbox1' , 'bbox2' , 'bbox3' , 'bbox4' , 'item_id'] 
        csv_writer = csv.DictWriter(new_file , feild names  = feild)
        csv_writer.writeheader()

with open('face.csv' , 'w') as new_file   
        feild  = ['Detection'  , 'person_id'] 
        csv_writer = csv.DictWriter(new_file , feild names  = feild)
        csv_writer.writeheader()

person = 1 
def track_person(frame):
     face = mp.solutions.face_mesh
     a = []  
     b = []  
     c = []
     result = [] 
     with face.FaceMesh(max_num_faces = 1  , 
                     refine_landmarks = True ,
                     min_detection_confidence = 0.5 , 
                     min_tracking_confidence = 0.5 ) as mesh :
          bgr = cv2.cvtColor(frame , cv2.COLOR_BGR2RGB)
          out =  mesh.process(bgr) 
          if out.multi_face_landmarks:
             for face_landmarks in out.multi_face_landmarks :
               for landmark in face_landmarks.landmark :
                 a.append(landmark.x)
                 b.append(landmark.y)
                 c.append(landmark.z) 
             for j in range(len(a)) :
               result.append(a[j] - min(a)) 
               result.append(b[j] - min(b)) 
               result.append(c[j] - min(c)) 

     with open('face.csv' , 'a') as file:
          csv_writer = csv.writer(file)
          for detections  in result : 
              csv_writer.writerow([detections, person ])
              person = person +1 

    
  
model = YOLO("yolov8n.pt")


with open('data.csv' , 'w') as new_file:
 feild  = ['bbox1' , 'bbox2' , 'bbox3' , 'bbox4' , 'id'] 
 csv_writer = csv.DictWriter(new_file , feild names  = feild)
 csv_writer.writeheader()


human_face = 0 
counter = 1 
webcam  = cv2.VideoCapture(0) 
last_time = -1 

while True: 
    ret , frame = webcam.read() 
    df = pd.read_csv('data.csv')

    if ret :
        if df.empty :
          bbox = image_detection(frame)
          with open('data.csv' , 'a') as file : 
           csv_writer = csv.writer(file)
           for box in bbox : 
             csv_writer.writerow([box[0] , box[1] , box[2] , box[3] , counter ]) 
             counter = counter +1 
             last_time  = time.time() 

        if last_time is not -1 and time.time()-last_time is 600  : 
           results = model(frame)
           for box in results[0].boxes: 
              if box.cls[0].item is 0  :
                 track_person(frame) 
                 human_face = 1 
                 break

           while human_face is 1 : 
              time.sleep(1)
              results = model(frame)
              human_face =  0 
              for box in results[0].boxes: 
                  if box.cls[0].item == 0 :
                    human_face = 1 
                    break
                  
           reset_csv() 
           with open('data.csv' , 'a') as file :
             csv_writer = csv.writer(file)
             for box in bbox : 
                 csv_writer.writerow([box[0] , box[1] , box[2] , box[3] , counter ]) 
                 counter = counter +1 
                 last_time  = time.time()
                 
webcam.release()
cv2.destroyAllWindows() 
          
           
           
           








SyntaxError: expected ':' (1902084934.py, line 8)